In [ ]:
# @title Github Data Loader, Tokenizer Training, and Dataset Builder

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd


except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import irishGPT as iGPT

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

r_t = iGPT.tokenizer.Regex_Tokenizer()
r_t.load('Datasets/shakespeare_vocab.json')

dataset = iGPT.dataset.IrishChatDataset('Datasets/shakespeare.txt', r_t)

Unable to download repo, either:
	A.) You're not on colab
	B.) It has already been cloned
/Users/wtheisen/Library/CloudStorage/GoogleDrive-wtheisen@nd.edu/My Drive/Artificial Intelligence/CSE 10124 - Building ChatGPT/Lectures
device: cpu


SyntaxError: invalid syntax (dataset.py, line 4)

In [ ]:
# @title Transformer Block

import math
import torch
import torch.nn.functional as F
from collections import namedtuple

from irishGPT.linear_layer import LinearLayer
from irishGPT.relu import ReLU

class Transformer:
    Grad_Info = namedtuple('Grad_Info', [
        # ---- Attention sub-block cache ----
        'x',           # (B, T, C) original input (before LN1)
        'x_hat',       # (B, T, C) normalized: (X - mu) / sigma
        'x_norm',      # (B, T, C) after gamma/beta scaling
        'sigma',       # (B, T, 1) std dev (for LN1 backward)
        'q', 'k', 'v',                    # (B, T, Dh)
        'scores',                          # (B, T, T)
        'attn',                            # (B, T, T)
        'attn_v',                          # (B, T, Dh)  = attn @ v
        # ---- FFN sub-block cache ----
        'y_attn',      # (B, T, C) output after attention residual (input to FFN sub-block)
        'y_attn_hat',  # (B, T, C) normalized for LN2
        'sigma2',      # (B, T, 1) std dev (for LN2 backward)
    ])

    def __init__(self, model_dim, head_dim=None, ffn_expand=4, device='cpu'):
        """
        Single-head self-attention + FFN with Pre-LayerNorm for stability.
        model_dim: C (embedding size)
        head_dim: Dh (defaults to model_dim)
        ffn_expand: expansion factor for the FFN hidden dimension (default 4x)
        """
        self.C = model_dim
        self.Dh = head_dim if head_dim is not None else model_dim
        assert self.Dh == self.C, "For this simple single-head block, set head_dim == model_dim."

        self.device = device
        self.eps = 1e-5

        # ---- LayerNorm 1 parameters (Pre-LN for Attention) ----
        self.gamma = torch.ones(self.C, device=self.device)
        self.beta  = torch.zeros(self.C, device=self.device)

        # ---- Attention weights (Xavier init, appropriate for attention) ----
        scale_in = math.sqrt(1.0 / self.C)
        self.Wq = torch.randn(self.Dh, self.C, device=self.device) * scale_in
        self.Wk = torch.randn(self.Dh, self.C, device=self.device) * scale_in
        self.Wv = torch.randn(self.Dh, self.C, device=self.device) * scale_in
        self.bq = torch.zeros(self.Dh, device=self.device)
        self.bk = torch.zeros(self.Dh, device=self.device)
        self.bv = torch.zeros(self.Dh, device=self.device)

        self.Wo = torch.randn(self.C, self.Dh, device=self.device) * scale_in
        self.bo = torch.zeros(self.C, device=self.device)

        # ---- LayerNorm 2 parameters (Pre-LN for FFN) ----
        self.gamma2 = torch.ones(self.C, device=self.device)
        self.beta2  = torch.zeros(self.C, device=self.device)

        # ---- FFN: Linear → ReLU → Linear (uses existing classes) ----
        ffn_dim = ffn_expand * model_dim
        self.ffn_linear1 = LinearLayer(model_dim, ffn_dim, device=self.device)
        self.ffn_relu    = ReLU()
        self.ffn_linear2 = LinearLayer(ffn_dim, model_dim, device=self.device)

        self.cache = None   # holds Grad_Info from the last forward

    def forward(self, X, key_pad_mask=None, causal=False):
        """
        X: (B, T, C)
        key_pad_mask: optional (B, T) bool; True where PAD token.
        causal: if True, apply lower-triangular mask.
        Returns: Z = FFN_residual(Attn_residual(X))  (B, T, C)
        """
        B, T, C = X.shape

        # ============================================================
        # Sub-block 1: LN → Self-Attention → Residual
        # ============================================================

        # ---- LayerNorm 1 ----
        mu    = X.mean(dim=-1, keepdim=True)                                          # (B, T, 1)
        sigma = torch.sqrt(X.var(dim=-1, keepdim=True, unbiased=False) + self.eps)    # (B, T, 1)
        X_hat  = (X - mu) / sigma                                                     # (B, T, C)
        X_norm = self.gamma * X_hat + self.beta                                        # (B, T, C)

        # ---- Self-Attention (computed from normalized input) ----
        Q = X_norm @ self.Wq.T + self.bq
        K = X_norm @ self.Wk.T + self.bk
        V = X_norm @ self.Wv.T + self.bv

        scores = (Q @ K.transpose(1, 2)) / math.sqrt(self.Dh)

        # Masks (optional)
        if key_pad_mask is not None:
            mask_k = key_pad_mask.unsqueeze(1).expand(B, T, T)
            scores = scores.masked_fill(mask_k, float('-inf'))
        if causal:
            tril = torch.tril(torch.ones(T, T, device=X.device, dtype=torch.bool))
            scores = scores.masked_fill(~tril, float('-inf'))

        attn = F.softmax(scores, dim=-1)   # (B, T, T)
        attn_v = attn @ V                  # (B, T, Dh)

        y_ctx = attn_v @ self.Wo.T + self.bo   # (B, T, C)
        Y_attn = X + y_ctx                     # residual uses ORIGINAL X

        # ============================================================
        # Sub-block 2: LN → FFN (Linear → ReLU → Linear) → Residual
        # ============================================================

        # ---- LayerNorm 2 ----
        mu2    = Y_attn.mean(dim=-1, keepdim=True)                                          # (B, T, 1)
        sigma2 = torch.sqrt(Y_attn.var(dim=-1, keepdim=True, unbiased=False) + self.eps)    # (B, T, 1)
        Y_attn_hat  = (Y_attn - mu2) / sigma2                                               # (B, T, C)
        Y_attn_norm = self.gamma2 * Y_attn_hat + self.beta2                                  # (B, T, C)

        # ---- FFN ----
        ffn_out = self.ffn_linear1.forward(Y_attn_norm)   # (B, T, 4C)
        ffn_out = self.ffn_relu.forward(ffn_out)           # (B, T, 4C)
        ffn_out = self.ffn_linear2.forward(ffn_out)        # (B, T, C)

        # ---- Residual ----
        Y = Y_attn + ffn_out

        # Cache for backward
        self.cache = Transformer.Grad_Info(
            x=X, x_hat=X_hat, x_norm=X_norm, sigma=sigma,
            q=Q, k=K, v=V, scores=scores, attn=attn, attn_v=attn_v,
            y_attn=Y_attn, y_attn_hat=Y_attn_hat, sigma2=sigma2
        )
        return Y

    def backward(self, dY, key_pad_mask=None, causal=False):
        """
        dY: gradient wrt output Y, shape (B, T, C)
        Returns: dX (B, T, C)
        """
        B, T, C = dY.shape
        D = C   # feature dimension for LayerNorm
        cache = self.cache

        # ============================================================
        # Sub-block 2 backward: FFN + LN2
        # ============================================================

        # Y = Y_attn + ffn_out → residual splits gradient
        dffn_out = dY.clone()          # gradient into FFN path
        dY_attn  = dY.clone()          # gradient through residual

        # ---- FFN backward (reverse order, classes handle their own caching) ----
        dffn_out = self.ffn_linear2.backward(dffn_out)    # (B, T, 4C)
        dffn_out = self.ffn_relu.backward(dffn_out)       # (B, T, 4C)
        dffn_out = self.ffn_linear1.backward(dffn_out)    # (B, T, C)
        dY_attn_norm = dffn_out

        # ---- LayerNorm 2 backward ----
        dY_attn_hat  = dY_attn_norm * self.gamma2
        self.dgamma2 = (dY_attn_norm * cache.y_attn_hat).sum(dim=(0, 1))    # (C,)
        self.dbeta2  = dY_attn_norm.sum(dim=(0, 1))                           # (C,)

        dY_attn_ln2 = (1.0 / (D * cache.sigma2)) * (
            D * dY_attn_hat
            - dY_attn_hat.sum(dim=-1, keepdim=True)
            - cache.y_attn_hat * (dY_attn_hat * cache.y_attn_hat).sum(dim=-1, keepdim=True)
        )

        # Total gradient w.r.t. Y_attn = residual + through-LN2
        dY_attn += dY_attn_ln2

        # ============================================================
        # Sub-block 1 backward: Attention + LN1
        # ============================================================

        # Y_attn = X + y_ctx → residual splits gradient
        dy_ctx = dY_attn.clone()        # gradient into attention path
        dX     = dY_attn.clone()        # gradient through residual

        # ---- Output projection backward ----
        self.dWo = dy_ctx.reshape(-1, C).T @ cache.attn_v.reshape(-1, self.Dh)
        self.dbo = dy_ctx.sum(dim=(0, 1))
        dattn_v  = dy_ctx @ self.Wo          # (B, T, Dh)

        # ---- Attention backward ----
        dv    = cache.attn.transpose(1, 2) @ dattn_v             # (B, T, Dh)
        dattn = dattn_v @ cache.v.transpose(1, 2)                # (B, T, T)

        # Softmax backward
        tmp     = (dattn * cache.attn).sum(dim=-1, keepdim=True) # (B, T, 1)
        dscores = (dattn - tmp) * cache.attn                     # (B, T, T)

        # Respect masks in backward
        if key_pad_mask is not None:
            mask_k = key_pad_mask.unsqueeze(1).expand(B, T, T)
            dscores = dscores.masked_fill(mask_k, 0.0)
        if causal:
            tril = torch.tril(torch.ones(T, T, device=dY.device, dtype=torch.bool))
            dscores = dscores.masked_fill(~tril, 0.0)

        factor = 1.0 / math.sqrt(self.Dh)
        dqk = dscores * factor                                   # (B, T, T)

        dq = dqk @ cache.k                                       # (B, T, Dh)
        dk = dqk.transpose(1, 2) @ cache.q                       # (B, T, Dh)

        # ---- Q/K/V parameter gradients (w.r.t. X_norm) ----
        self.dWq = dq.reshape(-1, self.Dh).T @ cache.x_norm.reshape(-1, self.C)
        self.dbq = dq.sum(dim=(0, 1))

        self.dWk = dk.reshape(-1, self.Dh).T @ cache.x_norm.reshape(-1, self.C)
        self.dbk = dk.sum(dim=(0, 1))

        self.dWv = dv.reshape(-1, self.Dh).T @ cache.x_norm.reshape(-1, self.C)
        self.dbv = dv.sum(dim=(0, 1))

        # Gradient w.r.t. X_norm from Q, K, V branches
        dX_norm = dq @ self.Wq + dk @ self.Wk + dv @ self.Wv    # (B, T, C)

        # ---- LayerNorm 1 backward ----
        dX_hat      = dX_norm * self.gamma
        self.dgamma = (dX_norm * cache.x_hat).sum(dim=(0, 1))    # (C,)
        self.dbeta  = dX_norm.sum(dim=(0, 1))                     # (C,)

        dX_ln = (1.0 / (D * cache.sigma)) * (
            D * dX_hat
            - dX_hat.sum(dim=-1, keepdim=True)
            - cache.x_hat * (dX_hat * cache.x_hat).sum(dim=-1, keepdim=True)
        )

        # Total: residual + attention-through-LN1
        dX += dX_ln

        return dX

    def update(self, lr):
        # SGD update for attention weights
        self.Wq -= lr * self.dWq; self.bq -= lr * self.dbq
        self.Wk -= lr * self.dWk; self.bk -= lr * self.dbk
        self.Wv -= lr * self.dWv; self.bv -= lr * self.dbv
        self.Wo -= lr * self.dWo; self.bo -= lr * self.dbo
        # SGD update for LayerNorm 1 parameters
        self.gamma -= lr * self.dgamma
        self.beta  -= lr * self.dbeta
        # SGD update for LayerNorm 2 parameters
        self.gamma2 -= lr * self.dgamma2
        self.beta2  -= lr * self.dbeta2
        # SGD update for FFN layers (LinearLayer.update handles its own W and b)
        self.ffn_linear1.update(lr)
        self.ffn_relu.update(lr)
        self.ffn_linear2.update(lr)


In [ ]:
# @title Generative Recurrent Neural Network

import torch
import torch.nn as nn
import torch.nn.functional as F

class SLM(nn.Module):
    def __init__(self, device='cpu'):
        super().__init__()
        self.device = device

        self.embedding = nn.Embedding(512, 128)
        self.transformer = Transformer(128, device=device)
        self.transformer2 = Transformer(128, device=device)
        self.transformer3 = Transformer(128, device=device)
        self.transformer4 = Transformer(128, device=device)
        #self.rnn = nn.RNN(128, 64, batch_first=True)
        self.output = nn.Linear(128, 512)

    def forward(self, x, masks=None):
        x = self.embedding(x)  # (B,T,E)

        out = self.transformer.forward(x, masks)
        out = self.transformer2.forward(out, masks)
        out = self.transformer3.forward(out, masks)
        out = self.transformer4.forward(out, masks)

        logits = self.output(out)      # (B,T,V)
        return logits

    @torch.no_grad()
    def _sample_logits(self, logits_last, temperature=0.8, top_k=512):
        if logits_last.dim() == 2:
            logits_last = logits_last[0]  # (V,)

        logits_last = logits_last / temperature

        vals, idx = torch.topk(logits_last, top_k)
        filtered = torch.full_like(logits_last, float("-inf"))
        filtered[idx] = vals
        logits_last = filtered.to(self.device)

        probs = F.softmax(logits_last, dim=-1)              # (V,)
        next_id = torch.multinomial(probs, num_samples=1)   # (1,)
        return int(next_id.item())

    @torch.no_grad()
    def generate(self, prompt_tokens, max_new_tokens=80, top_k=16, temperature=1.4):
        self.eval()
        x = prompt_tokens

        for _ in range(max_new_tokens):
            logits = self.forward(x)                 # recompute on full context
            next_logits = logits[:, -1, :]           # (1,V)
            next_id = self._sample_logits(next_logits, temperature, top_k)

            x_new = torch.tensor([[next_id]], dtype=torch.long, device=self.device)
            x = torch.cat([x, x_new], dim=1)

            if next_id == 258:                       # eos id
                break

        return x[0].tolist()

In [ ]:
# @title SLM Training Loop and Helper Function

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def train_slm(dataset, epochs=25, batch_size=64, lr=0.01, grad_clip=1.0):
    device = dataset.device
    V = len(dataset.tokenizer.vocab)

    model = SLM(device=device).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=dataset.collate,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss_sum = 0.0   # sum over tokens
        total_tokens = 0

        for X, Y_onehot, mask in loader:
            opt.zero_grad(set_to_none=True)

            ask = mask.bool()
            key_pad_mask = ~mask                      # transformer wants True=PAD

            logits = model.forward(X, key_pad_mask)   # <-- pass inverted mask
            log_probs = F.log_softmax(logits, dim=-1)

            per_pos_loss = -(Y_onehot * log_probs).sum(dim=-1)
            per_pos_loss = per_pos_loss * mask.float()  # keep real-token mask here

            loss_sum = per_pos_loss.sum()                        # scalar
            n_tokens = mask.sum().item()

            if n_tokens == 0:
                continue

            loss = loss_sum / n_tokens                           # average per real token
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            opt.step()

            total_loss_sum += loss_sum.item()
            total_tokens += n_tokens

        avg_loss = total_loss_sum / max(1, total_tokens)
        ppl = float(torch.exp(torch.tensor(avg_loss)))
        print(f"epoch {epoch:02d} | loss/token={avg_loss:.4f} | ppl={ppl:.2f}")
        ids = dataset.tokenizer.encode("<|sos|>Thou<|eos|>")[:-1]
        prompt_tokens = torch.tensor([ids], dtype=torch.long, device=device)  # (1,T)
        tokens = model.generate(prompt_tokens)
        print('Sample Generation for epoch:')
        print('Tokens:', tokens)
        print('Text:', r_t.decode(tokens), '\n')
        model.train()

    return model

In [ ]:
# @title SLM Training

slm = train_slm(dataset)

epoch 01 | loss/token=2.7427 | ppl=15.53
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 258]
Text: <|sos|>Thou<|eos|> 

epoch 02 | loss/token=2.2744 | ppl=9.72
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 258]
Text: <|sos|>Thou<|eos|> 

epoch 03 | loss/token=2.1863 | ppl=8.90
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 289, 394, 104, 262, 289, 394, 289, 394, 260, 289, 394, 289, 394, 368, 394, 354, 394, 289, 394, 277, 394, 277, 394, 258]
Text: <|sos|>Thou thushou thus thushe thus thus Mus Sus thus fus fus<|eos|> 

epoch 04 | loss/token=2.1413 | ppl=8.51
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 258]
Text: <|sos|>Thou<|eos|> 

epoch 05 | loss/token=2.1155 | ppl=8.29
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 258]
Text: <|sos|>Thou<|eos|> 

epoch 06 | loss/token=2.0972 | ppl=8.14
Sample Generation for epoch:
Tokens: [257, 84, 104, 262, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98, 98,

In [ ]:
# @title SLM Prompting

prompt="<|sos|>Thou<|eos|>"

ids = r_t.encode(prompt)[:-1]
prompt_tokens = torch.tensor([ids], dtype=torch.long, device=device)  # (1,T)

tokens = slm.generate(prompt_tokens)

print(r_t.decode(tokens))

<|sos|>Thou so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so so
